In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [ ]:
m = geemap.Map()
m.centerObject(point, 10)
#10 is zoom level, higher number means more zoomed in
m.addLayer(region, {'color': 'grey'}, 'Boundary')
m

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [6]:
def get_radar(start_date, end_date):
    return ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(region) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .median() \
        .clip(region) \
        .focal_median(50, 'circle', 'meters') # using 50 meter radius to reduce speckle noise

In [7]:
before = get_radar('2024-01-01', '2024-02-28') #during the dry, or winter season
after  = get_radar('2024-08-01', '2024-08-30') #during the flood season

In [9]:
# Formula: (After - Before).
# If land (-10dB) becomes water (-20dB), the result is -10.
difference = after.select('VH').subtract(before.select('VH'))

#we are using vh polarization as vh is more sensitive to water bodies

change_threshold = -6

In [10]:
flooded_pixels = difference.lt(change_threshold).selfMask()

In [11]:
water_threshold = -18 #standard water threshold
permanent_water = before.select('VH').lt(water_threshold)

In [12]:
#logic:  updateMask(permanent_water.not()) keeps pixels where permanent_water is False
final_flood = flooded_pixels.updateMask(permanent_water.Not())

m.addLayer(before, {'min': -25, 'max': -5}, 'Before (Dry)', False)
m.addLayer(after, {'min': -25, 'max': -5}, 'After (Wet)', False)
m.addLayer(final_flood, {'palette': ['red']}, 'Flooded Areas (Newly Added Water)')

m

Map(bottom=113517.0, center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg']…

In [13]:
#calculating the area of flooded regions

flood_stats = final_flood.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=region,
    scale=10,
    maxPixels=1e9
)
flooded_area_sqkm = ee.Number(flood_stats.get('VH')).divide(1e6).getInfo()

print(f"area with newly flooded regions: {flooded_area_sqkm} square kilometers")

area with newly flooded regions: 12.998623853505771 square kilometers
